# 🚀 SpeculativeConfig(mtp) の要求

必須項目・上限・model 指定の要否。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")

import re

sp = root / "config/speculative.py"
print("speculative.py exists:", sp.exists())
t = sp.read_text(errors="replace")

print("\n=== class / フィールド定義 ===")
for i, l in enumerate(t.splitlines(), 1):
    s = l.strip()
    if s.startswith("class ") or s.startswith("def ") or "mtp" in s.lower():
        print(f"  {i}: {s[:150]}")

print("\n=== 数値制約・必須項目 ===")
for i, l in enumerate(t.splitlines(), 1):
    s = l.strip()
    if ("gt=" in s or "ge=" in s or "le=" in s or "lt=" in s
            or "Field(" in s or "model:" in s or "num_speculative_tokens" in s):
        print(f"  {i}: {s[:150]}")

print("\n=== mtp のバリデーション（前後 12 行）===")
lines = t.splitlines()
for i, l in enumerate(lines, 1):
    if "mtp" in l.lower() and ("if " in l or "elif " in l or "raise" in l
                               or "assert" in l or "=" in l):
        a, b = max(0, i - 12), min(len(lines), i + 12)
        print(f"  --- line {i} ---")
        for j in range(a, b):
            print(f"   {j + 1}: {lines[j][:150]}")
        break

print("\n=== MTPModelTypes ===")
i = t.find("MTPModelTypes")
print(t[i:i + 900] if i > 0 else "not found")

print("\n=== 既定値（--speculative-config 未指定時）===")
for i, l in enumerate(t.splitlines(), 1):
    if "num_speculative_tokens" in l and "=" in l:
        print(f"  {i}: {l.strip()[:150]}")
